<div class="alert alert-success">

#### EECS 245: Mathematics for Machine Learning, Homework 5 Supplemental Notebook
    
# Random Matrices, Rank, and Cosine Similarity

### EECS 245, Fall 2026 at the University of Michigan
    
</div>

### Instructions

Most homeworks will have Jupyter Notebooks, like this one, designed to supplement the theoretical problems. 

To write and run code in this notebook, you have two options:

1. **Set up a Jupyter Notebook environment locally, and use `git` to clone our [course repository](https://github.com/eecs245/fa26-code/tree/main) (preferred).** For instructions on how to do this, see the [**Environment Setup**](https://eecs245.org/env-setup) page of the course website.
1. **Use the EECS 245 DataHub.** To do this, click the link provided in the Homework 5 PDF. Before doing so, read the instructions on the [**Environment Setup**](https://eecs245.org/env-setup/#option-2-using-the-eecs-245-datahub) page on how to use the DataHub.

**You do not need to submit this notebook to Pensive.** All code for Problem 8 is provided, and there is no autograder. Run the cells in order, examine the plots, and include four sets of handwritten explanations in your Homework 5 PDF. Tasks 1-4 are worth 1, 2, 3, and 2 points, respectively, for a total of 8 points.

Your submission time for Homework 5 is the **later of the two submission times**: your **Homework 5, Problem 1 Video Link** and your regular Homework 5 PDF.

In [ ]:
# Run this cell to get started.
import numpy as np
import pandas as pd

pd.options.plotting.backend = "plotly"

import plotly.express as px
import plotly.io as pio
import plotly.figure_factory as ff
from plotly.subplots import make_subplots

# Set default layout for all plotly figures
import plotly.graph_objs as go

custom_template = go.layout.Template(pio.templates["plotly_white"])
custom_template.layout.plot_bgcolor = "white"
custom_template.layout.paper_bgcolor = "white"
custom_template.layout.margin = dict(l=60, r=60, t=60, b=60)
custom_template.layout.width = 700
custom_template.layout.font = dict(
    family="Palatino Linotype, Palatino, serif",
    color="black"
)

pio.templates["custom"] = custom_template
pio.templates.default = "custom"

## Problem 8: Random Simulations

---

The `numpy.random` sub-module contains a number of functions for generating random numbers.

Each time you run the cell below, you should see a (possibly) different random integer, drawn from 1 through 6, inclusive. Think of it like rolling a die.

In [ ]:
np.random.randint(1, 7)


If you're curious as to how these random numbers are generated, check out the Wikipedia article on [Pseudorandom number generators](https://en.wikipedia.org/wiki/Pseudorandom_number_generator).

Each time you run the cell below, you'll see 3 randomly chosen names from our class roster (including the four staff members), drawn such that each name is equally likely to be chosen, and the same entry is not chosen twice.

Change `replace=False` to `replace=True` and run the cell several times. What do you notice?

In [ ]:
names = pd.read_csv('data/names.csv', header=None)[0].to_numpy()

np.random.choice(names, size=3, replace=False)

One application of random simulations is that they can be used to estimate the **probability** of an event whose exact probability is difficult to compute by hand. As an example, suppose we'd like to find the probability of seeing 60 or more heads in 100 flips of a fair coin. (This is a probability we can solve exactly, by the way, using knowledge from EECS 203 if you've taken it.) Run the following few cells.

In [ ]:
num_heads = []
for _ in range(10000):
    # Treat 0 as tails, 1 as heads
    flips = np.random.randint(0, 2, size=100)
    num_heads.append(np.sum(flips))

fig = px.histogram(num_heads, nbins=20, title='Distribution of Number of Heads in 100 Flips', histnorm='probability')
fig.update_layout(showlegend=False, xaxis_title='Number of Heads', yaxis_title='Probability')

In [ ]:
# This is an array with length 10000, in which each entry is either True or False.
# Change 60 to 50 or 40 and observe what happens to the array.
np.array(num_heads) >= 60

In [ ]:
# The mean of the Trues and Falses (which are treated as 1s and 0s, respectively) is the proportion of Trues,
# which is the estimated probability of seeing 60 or more heads in 100 flips of a fair coin.
# Change 60 to 50 or 40 and observe what happens to the estimated probability.
np.mean(np.array(num_heads) >= 60)

Each time you run the above three cells, the estimated probability is slightly different, since it depends on 10000 random experiments, each of which involved 100 random coin flips. There's a way to write the simulation above more compactly and without a `for`-loop, using `np.random.multinomial`, as we will see at the end of the semester when we cover probability in more detail.

For now, we'd like to introduce a relevant application: generating random **vectors and matrices**, i.e. (2D) arrays with random entries. Exploring randomly generated matrices is a good way to develop an intuition for various properties of matrices, including their rank and the linear independence and/or orthogonality of their rows and columns.

Run the cell below several times.

In [ ]:
# Run, then run again, then run again, then run again...
A_3_by_4 = np.random.randn(3, 4)
print('A = \n', A_3_by_4, '\n')
print('rank(A) =', np.linalg.matrix_rank(A_3_by_4))

Each time the cell above is run, `A` is a different random $3 \times 4$ matrix, whose elements are each chosen independently from a normal (Gaussian) distribution with mean 0 and standard deviation 1.

What you should notice is that almost every time, the rank of `A` is 3, even though – as we know – it's very well possible for a $3 \times 4$ matrix to have rank 2, 1, or 0.

<div class="alert alert-info" markdown="1">

### Task 1 (1 pt)

</div>

Run the cell below to define `random_matrix(n, d)`, which generates an `n`-by-`d` matrix with independent standard normal entries, and `distribution_of_ranks(n, d, num_trials)`, which plots the ranks of many such matrices. Then run the next cell to see the distribution for 1000 random 3-by-4 matrices.

In [ ]:
def random_matrix(n, d):
    return np.random.randn(n, d)

def distribution_of_ranks(n, d, num_trials=1000):
    ranks = [np.linalg.matrix_rank(random_matrix(n, d)) for _ in range(num_trials)]
    frequencies = pd.Series(ranks).value_counts(normalize=True).reindex(range(min(n, d) + 1), fill_value=0)
    fig = px.bar(x=frequencies.index, y=frequencies.values,
                 title=f'Ranks of Random Gaussian Matrices: {n} by {d}')
    fig.update_layout(xaxis_title='Rank', yaxis_title='Proportion of Trials', showlegend=False)
    fig.update_xaxes(dtick=1)
    return fig

In [ ]:
distribution_of_ranks(3, 4, 1000)

**In your PDF:** Write a one-sentence explanation of why the rank is almost always 3, even though a 3-by-4 matrix can have rank 2, 1, or 0.

<div class="alert alert-info" markdown="1">

### Task 2 (2 pts)

</div>

Run the cell below to define `random_product(n, d, p)`, which generates a random `n`-by-`d` matrix and a random `d`-by-`p` matrix with independent standard normal entries and returns their product. It also defines `distribution_of_ranks_for_products(n, d, p, num_trials)`, which plots the ranks of many such products.

Then run the next cell to see the distribution for 1000 products of a 3-by-4 matrix and a 4-by-5 matrix.

In [ ]:
def random_product(n, d, p):
    A = np.random.randn(n, d)
    B = np.random.randn(d, p)
    return A @ B

def distribution_of_ranks_for_products(n, d, p, num_trials=1000):
    ranks = [np.linalg.matrix_rank(random_product(n, d, p)) for _ in range(num_trials)]
    frequencies = pd.Series(ranks).value_counts(normalize=True).reindex(range(min(n, p) + 1), fill_value=0)
    fig = px.bar(x=frequencies.index, y=frequencies.values,
                 title=f'Ranks of Gaussian Matrix Products: ({n} by {d}) @ ({d} by {p})')
    fig.update_layout(xaxis_title='Rank', yaxis_title='Proportion of Trials', showlegend=False)
    fig.update_xaxes(dtick=1)
    return fig

In [ ]:
distribution_of_ranks_for_products(3, 4, 5, 1000)

**In your PDF:** Explain why the product almost always has the rank you observe. What is the largest possible rank of the product, and why?

<div class="alert alert-info" markdown="1">

### Task 3 (3 pts)

</div>

The randomly generated matrices in Tasks 1 and 2 were made up of arbitrary real numbers, drawn from a normal distribution.

Next, we'll explore what might happen if we generate matrices with **binary entries**, i.e. each entry is either 0 or 1. Might we see different results?

Run the provided code below. It defines the following two functions:
1. `random_square_binary_product(n)`, which generates two independent random `n`-by-`n` (square) matrices, each with independently chosen entries that are equally likely to be 0 or 1, and returns their product.
2. `distribution_of_ranks_for_square_binary_products(n, num_trials)`, which runs the first function `num_trials` times, computes the rank of each resulting matrix, and returns a `plotly` **bar chart** of the empirical distribution of the ranks.

For example, `distribution_of_ranks_for_square_binary_products(3, 1000)` returns a bar chart with 4 bars, one for each possible rank (0, 1, 2, 3).

In [ ]:
def random_square_binary_product(n):
    A = np.random.randint(0, 2, size=(n, n))
    B = np.random.randint(0, 2, size=(n, n))
    return A @ B

def distribution_of_ranks_for_square_binary_products(n, num_trials=10000):
    ranks = []
    for _ in range(num_trials):
        ranks.append(np.linalg.matrix_rank(random_square_binary_product(n)))

    # If ranks is a list of length num_trials, containing the rank of each product,
    # this will generate a bar chart showing the empirical distribution of the ranks.
    # If you're curious, ask your TA how this works!
    
    frequencies = pd.Series(ranks).value_counts(normalize=True).reindex(range(n + 1), fill_value=0)
    fig = px.bar(x=frequencies.index, y=frequencies.values,
                 title=f'Distribution of Ranks for Square Binary Products of size {n}x{n}')
    fig.update_layout(showlegend=False, xaxis_title='Rank', yaxis_title='Probability')
    fig.update_xaxes(dtick=1)
    return fig

After defining the functions above, run the cell below to look at the distributions of ranks for square binary products of size 3x3, 15x15, and 30x30.

In [ ]:
figs = [distribution_of_ranks_for_square_binary_products(i, 1000) for i in [3, 15, 30]]

combined_fig = make_subplots(
    rows=1, 
    cols=3,
    subplot_titles=['3x3', '15x15', '30x30']
)

for i, fig in enumerate(figs, start=1):
    for trace in fig.data:
        combined_fig.add_trace(trace, row=1, col=i)

combined_fig.update_xaxes(title_text='Rank', dtick=1)
combined_fig.update_yaxes(title_text='Probability', row=1, col=1)

# Update layout
combined_fig.update_layout(
    height=500,
    width=1000,
    showlegend=False,
)

combined_fig.show()


**In your PDF:** Address the following questions:

1. In the case of multiplying matrices with random real-valued entries, the rank was almost always the largest possible value it could have been. With binary entries, the rank seems to be more spread out. Why?
1. Why do the distributions seem to be getting more concentrated as the size of the matrix increases?

<div class="alert alert-info" markdown="1">

### Task 4 (2 pts)

</div>

The final idea we'll have you explore is a phenomenon associated with the **curse of dimensionality**. In particular, we'll look at how the cosine similarity between two random vectors in $\mathbb{R}^n$ behaves as $n$ increases.

Run the provided code below. It defines the following two functions:
1. `cosine_similarity_random_pair(n)`, which generates two independent random vectors in $\mathbb{R}^n$ using `np.random.randn(n)` and returns their cosine similarity.
1. `distribution_of_cosine_similarities(n, num_trials)`, which runs the first function `num_trials` times, and returns a `plotly` density curve of the cosine similarities. 

In [ ]:
def cosine_similarity_random_pair(n):
    A = np.random.randn(n)
    B = np.random.randn(n)
    return np.dot(A, B) / (np.linalg.norm(A) * np.linalg.norm(B))

def distribution_of_cosine_similarities(n, num_trials=10000):
    similarities = []
    for _ in range(num_trials):
        similarities.append(cosine_similarity_random_pair(n))

    # Assumes similarities is a list of length num_trials, containing the cosine similarity of each pair of random vectors.
    # This will generate a plot of the empirical distribution of the cosine similarities.
    # If you're curious, ask your TA how this works!

    hist = ff.create_distplot([similarities], ['Cosine Similarity'], show_hist=False, show_rug=True)
    hist.update_layout(title=f'n = {n}', xaxis_title='Cosine Similarity', yaxis_title='Density', showlegend=False, xaxis_range=[-1, 1])
    return hist

After defining the functions above, run the cell below to look at the distributions of cosine similarities for pairs of random vectors of size 5, 50, and 500.

In [ ]:
figs = [distribution_of_cosine_similarities(i, 1000) for i in [5, 50, 500]]

combined_fig = make_subplots(
    rows=1, 
    cols=3,
    subplot_titles=['n = 5', 'n = 50', 'n = 500']
)

for i, fig in enumerate(figs, start=1):
    for trace in fig.data:
        # Ensure each trace has xaxis range set to [-1, 1]
        trace.update(xaxis='x'+str(i) if i > 1 else 'x')
        combined_fig.add_trace(trace, row=1, col=i)
    # Set xaxis range for each subplot
    combined_fig.update_xaxes(range=[-1, 1], row=1, col=i)

combined_fig.update_xaxes(title_text='Cosine Similarity')
combined_fig.update_yaxes(title_text='Density', row=1, col=1)

# Update layout
combined_fig.update_layout(
    height=500,
    width=1200,
    showlegend=False,
)

combined_fig.show()

You'll notice that the density curves are all centered around 0. What's changing is their spread, or standard deviation.

Run the cell below, then move the slider to compare the cosine-similarity distributions for different numbers of components.

In [ ]:
# Run this cell after running the cells above.
# Move the slider to compare cosine similarities in different dimensions.
dimensions = [5, 25, 50, 100, 200]
slider_fig = go.Figure()
for n in dimensions:
    curve = distribution_of_cosine_similarities(n, 1000).data[0]
    slider_fig.add_trace(go.Scatter(x=curve.x, y=curve.y, mode='lines',
                                   name=f'n = {n}', visible=False))
slider_fig.data[0].visible = True
steps = []
for i, n in enumerate(dimensions):
    steps.append(dict(method='update', label=str(n),
                      args=[{'visible': [j == i for j in range(len(dimensions))]},
                            {'title': f'Cosine Similarities: n = {n}'}]))
slider_fig.update_layout(
    title='Cosine Similarities: n = 5', showlegend=False,
    xaxis_title='Cosine Similarity', yaxis_title='Density', xaxis_range=[-1, 1],
    sliders=[dict(active=0, currentvalue={'prefix': 'Number of components: '}, steps=steps)])
slider_fig.show()

**In your PDF:** Explain the significance of the results above. As $n$ increases, what becomes more and more likely about the relationship between two random vectors in $\mathbb{R}^n$? What is the significance of the term "the curse of dimensionality"?

## Finish Line 🏁

You do not need to submit this notebook. Under Problem 8 in your regular Homework 5 PDF, include **four sets of handwritten explanations**:

- **Task 1 (1 point):** Your explanation of the Gaussian matrix ranks.
- **Task 2 (2 points):** Your explanation of the Gaussian matrix product ranks.
- **Task 3 (3 points):** Your responses to both questions about binary matrix product ranks.
- **Task 4 (2 points):** Your responses to both questions about cosine similarity.

No screenshots of plots, code, or other outputs are required for Problem 8.

Your submission time for Homework 5 is the **later of the two submission times**: your **Homework 5, Problem 1 Video Link** and your regular Homework 5 PDF.